# WarheadVerify: arms-control claim verification (corpus, baselines, cross-encoder, LLM agent)

Companion notebook for [MHKhan50/warheadverify-agentbench](https://github.com/MHKhan50/warheadverify-agentbench).
Claims about nuclear arms-control treaty status (PTBT, NPT, CTBT, TPNW) are verified against yearly status records from
the UNODA Treaties Database (published by Our World in Data, CC BY 4.0). Public treaty status only; no weapons information.

Kaggle settings: **GPU T4**, **Internet on**. *Save Version → Save & Run All*. A smoke test runs first; the full run takes
about 1 hour. Outputs: `/kaggle/working/reports/` and `/kaggle/working/warheadverify_reports.zip`.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/MHKhan50/warheadverify-agentbench.git"
os.chdir("/kaggle/working")
if not os.path.exists("warheadverify-agentbench"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
os.chdir("warheadverify-agentbench")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
RAW, DATA, OUT = "/kaggle/working/data/raw", "/kaggle/working/data/processed", "/kaggle/working/reports"
def wv(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "warheadverify.cli", *args, "--raw-dir", RAW, "--data-dir", DATA, "--out", OUT],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

## 1. Download the status tables (manifest with SHA-256) and build the corpus

In [ ]:
wv("download")
wv("build")

## 2. CPU: retrieval, lexical baselines, lookup agent

In [ ]:
wv("cpu")

## 3. Smoke test of the GPU step (1 epoch, 1 seed, 30 LLM claims), then the full run

In [ ]:
wv("gpu", "--seeds", "0", "--epochs", "1", "--llm-claims", "30")
wv("gpu")

## 4. Summary and zip

In [ ]:
wv("summary")
import shutil, glob
shutil.copytree(DATA, OUT + "/corpus", dirs_exist_ok=True)
shutil.copy(RAW + "/manifest.json", OUT + "/corpus/manifest.json")
shutil.make_archive("/kaggle/working/warheadverify_reports", "zip", OUT)